----------------------------------------------------------------------------------------------------

# ***SQL (Structured Query Language)***

----------------------------------------------------------------------------------------------------

O SQL é a linguagem usada para interagir com bancos de dados relacionais. Com SQL, podemos criar tabelas, inseir, atualziar e deletar registros (esse processo é chamado de CRUD), bem como executar consutlas para buscar dados.

----------------------------------------------------------------------------------------------------

# ***1. Aplicando SQL - Utilizando CRUD:*** 

## ***1.1. C - CREATE***

In [ ]:
-- Cria um novo banco de dados
CREATE DATABASE loja;

-- Cria uma tabela para armazenar informações de produtos
CREATE TABLE produtos (id INTEGER PRIMARY KEY AUTOINCREMENT, nome VARCHAR(100), preco DECIMAL);

-- Inclui um novo produto
INSERT INTO produtos (nome, preco) VALUES ('Curso de Python', 258.00);

### ***1.1.1. C - CREATE -> Criação de tabelas***

In [ ]:
-- Criar tabela de produtos
CREATE TABLE produtos (
  id       INTEGER PRIMARY KEY AUTOINCREMENT,
  nome     TEXT    NOT NULL,
  preco    REAL    NOT NULL,
  estoque  INTEGER DEFAULT 0,
  categoria TEXT
);

-- Criar tabela de clientes
CREATE TABLE clientes (
  id    INTEGER PRIMARY KEY AUTOINCREMENT,
  nome  TEXT NOT NULL,
  email TEXT UNIQUE NOT NULL
);

## ***1.2. R - READ***

In [ ]:
-- Lista todos os produtos
SELECT FROM produtos;

## ***1.3. U - UPDATE***

In [ ]:
-- Atualiza o produto com id informado
UPDATE produtos SET nome Curso de Python para iniciantes WHERE id = 1;

### ***1.3. D - DELETE***

In [ ]:
-- Exclui o produto con id informado
DELETE FROM produtos WHERE id = 1;

----------------------------------------------------------------------------------------------------

# ***2. Criação de um projeto*** 

----------------------------------------------------------------------------------------------------

## ***2.1. Módulo 1 — Fundamentos***

### ***2.1.1. Criando seu primeiro banco — uma loja virtual simples***

In [ ]:
-- Criar tabela de produtos
CREATE TABLE produtos (
  id       INTEGER PRIMARY KEY AUTOINCREMENT,
  nome     TEXT    NOT NULL,
  preco    REAL    NOT NULL,
  estoque  INTEGER DEFAULT 0,
  categoria TEXT
);

-- Criar tabela de clientes
CREATE TABLE clientes (
  id    INTEGER PRIMARY KEY AUTOINCREMENT,
  nome  TEXT NOT NULL,
  email TEXT UNIQUE NOT NULL
);

No SQLite os tipos principais são: INTEGER, TEXT, REAL, BLOB, NULL

### ***2.1.2. Inserindo Dados (INSERT)***

In [ ]:
-- Inserindo um produto
INSERT INTO produtos (nome, preco, estoque, categoria)
VALUES ('Camiseta Azul', 49.90, 100, 'Roupas');

-- Inserindo vários de uma vez
INSERT INTO produtos (nome, preco, estoque, categoria) VALUES
  ('Calça Jeans',   129.90, 50,  'Roupas'),
  ('Tênis Branco',  199.90, 30,  'Calçados'),
  ('Boné Preto',     39.90, 200, 'Acessórios'),
  ('Mochila Verde', 159.90, 25,  'Acessórios');

-- Inserindo clientes
INSERT INTO clientes (nome, email) VALUES
  ('Ana Silva',   'ana@email.com'),
  ('Bruno Costa', 'bruno@email.com'),
  ('Carla Souza', 'carla@email.com');

### ***2.1.3. Consultando Dados (SELECT)***

In [ ]:
-- Buscar tudo
SELECT * FROM produtos;

-- Buscar só algumas colunas
SELECT nome, preco FROM produtos;

-- Filtrar com WHERE
SELECT * FROM produtos WHERE categoria = 'Roupas';

-- Maior que / Menor que
SELECT * FROM produtos WHERE preco > 100;

-- Múltiplas condições
SELECT * FROM produtos
WHERE categoria = 'Acessórios' AND preco < 100;

-- Busca por texto parcial (LIKE)
SELECT * FROM produtos WHERE nome LIKE '%Azul%';

-- Ordenar resultados
SELECT * FROM produtos ORDER BY preco DESC;

-- Limitar resultados (útil para paginação em sites!)
SELECT * FROM produtos ORDER BY preco ASC LIMIT 3;

### ***2.1.4. Atualizando e Deletando***

In [ ]:
-- Atualizar um produto
UPDATE produtos SET preco = 44.90 WHERE id = 1;

-- Atualizar estoque após uma venda
UPDATE produtos SET estoque = estoque - 1 WHERE id = 1;

-- Deletar um registro
DELETE FROM produtos WHERE id = 5;

-- CUIDADO: sem WHERE, afeta TODOS os registros!
-- DELETE FROM produtos;  → apaga tudo!

----------------------------------------------------------------------------------------------------

## ***2.2. Módulo 2 — Relacionamentos***

### ***2.2.1. Criando tabelas relacionadas — sistema de pedidos***

In [ ]:
-- Tabela de pedidos
CREATE TABLE pedidos (
  id         INTEGER PRIMARY KEY AUTOINCREMENT,
  cliente_id INTEGER NOT NULL,
  data       TEXT    DEFAULT (DATE('now')),
  status     TEXT    DEFAULT 'pendente',
  FOREIGN KEY (cliente_id) REFERENCES clientes(id)
);

-- Tabela intermediária: itens do pedido (N:M)
CREATE TABLE itens_pedido (
  pedido_id  INTEGER,
  produto_id INTEGER,
  quantidade INTEGER NOT NULL DEFAULT 1,
  PRIMARY KEY (pedido_id, produto_id),
  FOREIGN KEY (pedido_id)  REFERENCES pedidos(id),
  FOREIGN KEY (produto_id) REFERENCES produtos(id)
);

-- Inserindo pedidos
INSERT INTO pedidos (cliente_id) VALUES (1), (2);

INSERT INTO itens_pedido (pedido_id, produto_id, quantidade) VALUES
  (1, 1, 2),  -- pedido 1: 2x Camiseta
  (1, 3, 1),  -- pedido 1: 1x Tênis
  (2, 2, 1);  -- pedido 2: 1x Calça

### ***2.2.3. JOIN — Cruzando tabelas (fundamental para web)***

In [ ]:
-- Ver pedidos com nome do cliente
SELECT
  pedidos.id       AS pedido,
  clientes.nome    AS cliente,
  pedidos.status,
  pedidos.data
FROM pedidos
JOIN clientes ON clientes.id = pedidos.cliente_id;

-- Ver todos os itens de todos os pedidos (3 tabelas!)
SELECT
  c.nome          AS cliente,
  p.id            AS pedido,
  pr.nome         AS produto,
  ip.quantidade,
  pr.preco,
  (ip.quantidade * pr.preco) AS subtotal
FROM itens_pedido ip
JOIN pedidos  p  ON p.id  = ip.pedido_id
JOIN clientes c  ON c.id  = p.cliente_id
JOIN produtos pr ON pr.id = ip.produto_id;

-- Clientes que NUNCA fizeram pedido (LEFT JOIN)
SELECT c.nome
FROM clientes c
LEFT JOIN pedidos p ON p.cliente_id = c.id
WHERE p.id IS NULL;

----------------------------------------------------------------------------------------------------

## ***2.3. Módulo 3 — Agregações e Consultas Úteis para Web***

### ***2.3.1. Agregações e Consultas***

In [ ]:
-- Total de produtos por categoria
SELECT categoria, COUNT(*) AS total
FROM produtos
GROUP BY categoria;

-- Ticket médio por cliente
SELECT
  c.nome,
  COUNT(p.id)                          AS total_pedidos,
  SUM(ip.quantidade * pr.preco)        AS valor_total,
  AVG(ip.quantidade * pr.preco)        AS ticket_medio
FROM clientes c
JOIN pedidos p      ON p.cliente_id = c.id
JOIN itens_pedido ip ON ip.pedido_id = p.id
JOIN produtos pr    ON pr.id = ip.produto_id
GROUP BY c.id, c.nome
ORDER BY valor_total DESC;

-- Produtos com estoque baixo (alerta no painel admin)
SELECT nome, estoque
FROM produtos
WHERE estoque < 30
ORDER BY estoque ASC;

-- Busca de produto por nome (campo de pesquisa no site)
SELECT * FROM produtos
WHERE nome LIKE '%' || 'cam' || '%';  -- busca dinâmica

----------------------------------------------------------------------------------------------------

## ***2.4. Módulo 4 — SQLite na Prática Web***

### ***2.4.1. Conexão SQLite***

No terminal, digite:

`pip install fastapi uvicorn`

Crie um arquivo chamado 'database.py'

In [ ]:
import sqlite3

DB_PATH = "loja.db"

def get_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row  # retorna como dicionário
    return conn

def init_db():
    conn = get_db()
    conn.executescript("""
        CREATE TABLE IF NOT EXISTS produtos (
            id        INTEGER PRIMARY KEY AUTOINCREMENT,
            nome      TEXT    NOT NULL,
            preco     REAL    NOT NULL,
            estoque   INTEGER DEFAULT 0,
            categoria TEXT
        );

        CREATE TABLE IF NOT EXISTS clientes (
            id    INTEGER PRIMARY KEY AUTOINCREMENT,
            nome  TEXT NOT NULL,
            email TEXT UNIQUE NOT NULL
        );
    """)
    conn.commit()
    conn.close()

Crie um arquivo chamado 'main.py'

In [ ]:
from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel
from typing import Optional
from database import get_db, init_db

app = FastAPI(title="Loja API", version="1.0.0")

# Inicializa o banco ao subir a aplicação
@app.on_event("startup")
def startup():
    init_db()

#  Modelos (Pydantic valida os dados!)

class ProdutoCreate(BaseModel):
    nome:      str
    preco:     float
    estoque:   int = 0
    categoria: Optional[str] = None

class ProdutoUpdate(BaseModel):
    nome:      Optional[str]   = None
    preco:     Optional[float] = None
    estoque:   Optional[int]   = None
    categoria: Optional[str]   = None

class ClienteCreate(BaseModel):
    nome:  str
    email: str


# Rotas de Produtos


# GET /produtos → lista todos os produtos
@app.get("/produtos")
def listar_produtos():
    conn = get_db()
    produtos = conn.execute("SELECT * FROM produtos").fetchall()
    conn.close()
    return [dict(p) for p in produtos]


# GET /produtos/busca?q=camiseta → busca por nome
@app.get("/produtos/busca")
def buscar_produtos(q: str = Query(..., min_length=1, description="Termo de busca")):
    conn = get_db()
    produtos = conn.execute(
        "SELECT * FROM produtos WHERE nome LIKE ?",
        (f"%{q}%",)  # ✅ Parâmetro seguro — sem SQL Injection!
    ).fetchall()
    conn.close()
    return [dict(p) for p in produtos]


# GET /produtos/{id} → busca produto por ID
@app.get("/produtos/{produto_id}")
def buscar_produto(produto_id: int):
    conn = get_db()
    produto = conn.execute(
        "SELECT * FROM produtos WHERE id = ?", (produto_id,)
    ).fetchone()
    conn.close()

    if not produto:
        raise HTTPException(status_code=404, detail="Produto não encontrado")

    return dict(produto)


# POST /produtos → cadastrar novo produto
@app.post("/produtos", status_code=201)
def criar_produto(produto: ProdutoCreate):
    conn = get_db()
    cursor = conn.execute(
        "INSERT INTO produtos (nome, preco, estoque, categoria) VALUES (?, ?, ?, ?)",
        (produto.nome, produto.preco, produto.estoque, produto.categoria)
    )
    conn.commit()
    novo_id = cursor.lastrowid
    conn.close()
    return {"mensagem": "Produto criado!", "id": novo_id}


# PATCH /produtos/{id} → atualizar produto parcialmente
@app.patch("/produtos/{produto_id}")
def atualizar_produto(produto_id: int, dados: ProdutoUpdate):
    conn = get_db()

    # Verifica se existe
    produto = conn.execute(
        "SELECT * FROM produtos WHERE id = ?", (produto_id,)
    ).fetchone()

    if not produto:
        conn.close()
        raise HTTPException(status_code=404, detail="Produto não encontrado")

    # Atualiza apenas os campos enviados
    campos = {k: v for k, v in dados.model_dump().items() if v is not None}
    for campo, valor in campos.items():
        conn.execute(
            f"UPDATE produtos SET {campo} = ? WHERE id = ?", (valor, produto_id)
        )

    conn.commit()
    conn.close()
    return {"mensagem": "Produto atualizado!"}


# DELETE /produtos/{id} → remover produto
@app.delete("/produtos/{produto_id}")
def deletar_produto(produto_id: int):
    conn = get_db()
    produto = conn.execute(
        "SELECT * FROM produtos WHERE id = ?", (produto_id,)
    ).fetchone()

    if not produto:
        conn.close()
        raise HTTPException(status_code=404, detail="Produto não encontrado")

    conn.execute("DELETE FROM produtos WHERE id = ?", (produto_id,))
    conn.commit()
    conn.close()
    return {"mensagem": "Produto deletado!"}


# Rotas de Clientes

@app.get("/clientes")
def listar_clientes():
    conn = get_db()
    clientes = conn.execute("SELECT * FROM clientes").fetchall()
    conn.close()
    return [dict(c) for c in clientes]


@app.post("/clientes", status_code=201)
def criar_cliente(cliente: ClienteCreate):
    conn = get_db()
    try:
        cursor = conn.execute(
            "INSERT INTO clientes (nome, email) VALUES (?, ?)",
            (cliente.nome, cliente.email)
        )
        conn.commit()
        novo_id = cursor.lastrowid
        return {"mensagem": "Cliente criado!", "id": novo_id}
    except Exception:
        # Email duplicado (UNIQUE constraint)
        raise HTTPException(status_code=400, detail="E-mail já cadastrado")
    finally:
        conn.close()

----------------------------------------------------------------------------------------------------